In [37]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


In [38]:
df_CoA = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/chart_of_account_OB.csv')
df_customer = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/customer_table.csv')
df_employee = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/employee_table.csv')
df_Master_txn = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/Master_txn_table.csv')
df_payment_method = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/payment_method.csv')
df_product_service = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/product_service_table.csv')
df_vendor = pd.read_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/vendor_table.csv')

In [39]:
dfs = {
    "chart_of_account_OB": df_CoA,
    "customer_table": df_customer,
    "employee_table": df_employee,
    "Master_txn_table": df_Master_txn,
    "payment_method": df_payment_method,
    "product_service_table": df_product_service,
    "vendor_table": df_vendor
}

list(dfs.keys())


['chart_of_account_OB',
 'customer_table',
 'employee_table',
 'Master_txn_table',
 'payment_method',
 'product_service_table',
 'vendor_table']

In [40]:
inventory = []

for name, df in dfs.items():
    inventory.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

pd.DataFrame(inventory)


,dataset,rows,columns
0,chart_of_account_OB,2430,5
1,customer_table,50000,13
2,employee_table,50000,7
3,Master_txn_table,810059,32
4,payment_method,189,4
5,product_service_table,270,4
6,vendor_table,100000,8


In [41]:
for name, df in dfs.items():
    print(f"\n📂 Dataset: {name}")
    print("-" * 60)
    print(df.columns.tolist())



📂 Dataset: chart_of_account_OB
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Account name', 'Account Full Name', 'Account type']

📂 Dataset: customer_table
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Customer name', 'Customer full name', 'Billing address', 'Billing city', 'Billing state', 'Billing ZIP code', 'Shipping address', 'Shipping city', 'Shipping state', 'Shipping ZIP code', 'Balance']

📂 Dataset: employee_table
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Employee name', 'Employee ID', 'Hire date', 'Billing rate', 'Deleted']

📂 Dataset: Master_txn_table
------------------------------------------------------------
['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'business Id', 'Transaction ID', 'Transaction date', 'Transaction type', 'Amount', 'Created date', 'Created user', 'Account', 'A/R paid', 'A/P paid', 'Due date', 'Op

In [46]:
import pandas as pd

# Load datasets
industry_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry_Details.csv")
coa_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/chart_of_account_OB.csv")
ps_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/product_service_table.csv")

# Check actual column names
print("industry_df columns:", industry_df.columns.tolist())
print("coa_df columns:", coa_df.columns.tolist())
print("ps_df columns:", ps_df.columns.tolist())

# Clean all column names to lowercase with underscores
industry_df.columns = industry_df.columns.str.strip().str.lower().str.replace(' ', '_')
coa_df.columns = coa_df.columns.str.strip().str.lower().str.replace(' ', '_')
ps_df.columns = ps_df.columns.str.strip().str.lower().str.replace(' ', '_')

print("\nCleaned columns:")
print("industry_df:", industry_df.columns.tolist())
print("coa_df:", coa_df.columns.tolist())
print("ps_df:", ps_df.columns.tolist())

# Normalize text
def normalize(text):
    return str(text).strip().lower()

industry_df["account_name"] = industry_df["account_name"].apply(normalize)
industry_df["product_service"] = industry_df["product_service"].apply(normalize)

coa_df["account_name"] = coa_df["account_name"].apply(normalize)
ps_df["product_service"] = ps_df["product_service"].apply(normalize)

# ... rest of your merging logic ...
industry_df["industry"] = industry_df["industry"].str.strip()

# ============================
# 3. Evidence from Accounts
# ============================

account_evidence = coa_df.merge(
    industry_df[["industry", "account_name"]],
    on="account_name",
    how="inner"
)[["business_id", "industry"]]

account_evidence["evidence"] = "account"

# ============================
# 4. Evidence from Product / Services
# ============================

ps_evidence = ps_df.merge(
    industry_df[["industry", "product_service"]],
    on="product_service",
    how="inner"
)[["business_id", "industry"]]

ps_evidence["evidence"] = "product_service"

# ============================
# 5. Combine Evidence
# ============================

combined = pd.concat(
    [account_evidence, ps_evidence],
    ignore_index=True
)

# ============================
# 6. Aggregate evidence counts
# ============================

industry_counts = (
    combined
    .groupby(["business_id", "industry"])
    .size()
    .reset_index(name="evidence_count")
)

# ============================
# 7. Select dominant industry
# ============================

dominant_industry = (
    industry_counts
    .sort_values(
        ["business_id", "evidence_count"],
        ascending=[True, False]
    )
    .groupby("business_id")
    .first()
    .reset_index()
)

# ============================
# 8. (Optional but recommended) confidence score
# ============================

total_counts = (
    industry_counts
    .groupby("business_id")["evidence_count"]
    .sum()
    .reset_index(name="total_evidence")
)

dominant_industry = dominant_industry.merge(
    total_counts,
    on="business_id"
)

dominant_industry["confidence"] = (
    dominant_industry["evidence_count"]
    / dominant_industry["total_evidence"]
).round(2)

# ============================
# 9. Final mapping
# ============================

business_industry_map = dominant_industry[
    ["business_id", "industry", "confidence"]
]

business_industry_map.to_csv(
    "business_id_to_industry.csv",
    index=False
)

print("✅ Business → Industry mapping created")


industry_df columns: ['Industry', 'Business', 'Account Name', 'Account Type', 'Product_Service', 'Product_Service_Type']
coa_df columns: ['Unnamed: 0', 'Business Id', 'Account name', 'Account Full Name', 'Account type']
ps_df columns: ['Unnamed: 0', 'Business Id', 'Product_Service', 'Product_Service_Type']

Cleaned columns:
industry_df: ['industry', 'business', 'account_name', 'account_type', 'product_service', 'product_service_type']
coa_df: ['unnamed:_0', 'business_id', 'account_name', 'account_full_name', 'account_type']
ps_df: ['unnamed:_0', 'business_id', 'product_service', 'product_service_type']
✅ Business → Industry mapping created


In [ ]:
# Create a presence/absence matrix of columns across datasets

all_columns = sorted(set(col for df in dfs.values() for col in df.columns))

presence_df = pd.DataFrame(index=all_columns)

for name, df in dfs.items():
    presence_df[name] = presence_df.index.isin(df.columns)

presence_df


In [10]:

print(dfs['chart_of_account_OB'].columns.tolist())
print("Cleaning dataframes...")

for name, df in dfs.items():
    # 1. Remove "Unnamed" index columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    import pandas as pd

# 1. Load the Transactions Table (Where the actual activity is)
df_txn = dfs['Master_txn_table']

print("Analyzing Transaction Activity per Business...")

# 2. Define a function to score each business based on usage
def profile_business_activity(df, bid):
    # Filter for this business
    subset = df[df['business_id'] == bid]
    
    # Get the list of Accounts and Products actually used in transactions
    used_accounts = subset['account'].astype(str).str.lower().unique()
    used_products = subset['product_service'].astype(str).str.lower().unique()
    
    # Combine into one searchable text
    activity_text = " ".join(used_accounts) + " " + " ".join(used_products)
    
    # --- SCORING SYSTEM ---
    scores = {'Retail': 0, 'Service': 0, 'Real_Estate': 0}
    
    # Retail Signals (Inventory, Sales of Product)
    if any(x in activity_text for x in ['inventory', 'product', 'shipping', 'cogs', 'goods sold']):
        scores['Retail'] += 5
        
    # Service Signals (Consulting, Hours, Design)
    if any(x in activity_text for x in ['consulting', 'hours', 'service', 'design', 'legal']):
        scores['Service'] += 5
        
    # Real Estate Signals (Job Materials, Subcontractors, WIP)
    if any(x in activity_text for x in ['job materials', 'wip', 'subcontractor', 'concrete', 'roofing']):
        scores['Real_Estate'] += 5
        
    # Tie-Breaker Logic    
    # If it uses "Job Materials", it's almost certainly Construction/Real Estate
    if 'job materials' in activity_text:
        return "Real_Estate"
    
    # Get the category with the highest score
    best_match = max(scores, key=scores.get)
    
    # If no score, return Undefined
    if scores[best_match] == 0:
        return "Undefined"
        
    return best_match

# 3. Analyze all 27 businesses
results = []
unique_bids = df_txn['business_id'].unique()

for bid in unique_bids:
    industry = profile_business_activity(df_txn, bid)
    results.append({'business_id': bid, 'industry': industry})

# 4. Convert to DataFrame and Show
df_results = pd.DataFrame(results)

print("\n--- TRUE Industry Distribution (Based on Transactions) ---")
print(df_results['industry'].value_counts())

print("\n--- Detailed Assignment ---")
print(df_results)
    # 2. Standardize column names (lowercase, no spaces)
    # This turns "Business Id" -> "business_id"
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # 3. Replace the "--" placeholders with actual NaN/Null
    df.replace('--', np.nan, inplace=True)
    
    # Update the dictionary
    dfs[name] = df

print("Cleaning complete. New column format example:")
print(dfs['chart_of_account_OB'].columns.tolist())

IndentationError: unexpected indent (2323339507.py, line 73)

In [11]:
for name, df in dfs.items():
    print(f"\n📂 Dataset: {name}")
    print("-" * 60)
    print(df.columns.tolist())



📂 Dataset: chart_of_account_OB
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Account name', 'Account Full Name', 'Account type']

📂 Dataset: customer_table
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Customer name', 'Customer full name', 'Billing address', 'Billing city', 'Billing state', 'Billing ZIP code', 'Shipping address', 'Shipping city', 'Shipping state', 'Shipping ZIP code', 'Balance']

📂 Dataset: employee_table
------------------------------------------------------------
['Unnamed: 0', 'Business Id', 'Employee name', 'Employee ID', 'Hire date', 'Billing rate', 'Deleted']

📂 Dataset: Master_txn_table
------------------------------------------------------------
['Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'business Id', 'Transaction ID', 'Transaction date', 'Transaction type', 'Amount', 'Created date', 'Created user', 'Account', 'A/R paid', 'A/P paid', 'Due date', 'Op

In [43]:
# Extract the Chart of Accounts dataframe
df_coa = dfs['chart_of_account_OB']

# Group by Business ID and combine all account names into one long string
# This creates a "Bag of Words" for each company
business_profiles = df_coa.groupby('business_id')['account_name'].apply(
    lambda x: ' '.join(x.astype(str)).lower()
).reset_index()

business_profiles.columns = ['business_id', 'account_keywords']

print(f"Analyzed {len(business_profiles)} unique businesses.")
print(business_profiles.head(30))

Analyzed 27 unique businesses.
    business_id                                   account_keywords
0             2  bookkeeper lawyer maintenance and repair build...
1             3  bookkeeper lawyer maintenance and repair build...
2             4  bookkeeper lawyer maintenance and repair build...
3             5  bookkeeper lawyer maintenance and repair build...
4             6  bookkeeper lawyer maintenance and repair build...
5             7  bookkeeper lawyer maintenance and repair build...
6             8  bookkeeper lawyer maintenance and repair build...
7             9  bookkeeper lawyer maintenance and repair build...
8            10  bookkeeper lawyer maintenance and repair build...
9            11  bookkeeper lawyer maintenance and repair build...
10           12  bookkeeper lawyer maintenance and repair build...
11           13  bookkeeper lawyer maintenance and repair build...
12           14  bookkeeper lawyer maintenance and repair build...
13           15  bookkeeper law

In [44]:
# Define the classification logic again
def classify_industry(keywords):
    # RETAIL: Look for inventory stuff
    if any(x in keywords for x in ['inventory', 'cost of goods', 'cogs', 'shipping income']):
        return "Retail"
    
    # REAL ESTATE: Look for construction stuff
    elif any(x in keywords for x in ['job materials', 'construction', 'wip', 'subcontractors']):
        return "Real Estate"
    
    # SERVICE: Look for consulting/professional fees (and NO inventory)
    elif any(x in keywords for x in ['consulting', 'professional fees', 'service income', 'billable']):
        return "Service"
        
    return "General/Other"

# Apply to all 27 businesses
business_profiles['predicted_industry'] = business_profiles['account_keywords'].apply(classify_industry)

# Show me the count of each industry found
print("\n--- Industry Distribution (Total 27) ---")
print(business_profiles['predicted_industry'].value_counts())

# Show exactly which ID belongs to which Industry
print("\n--- Detailed List ---")
print(business_profiles[['business_id', 'predicted_industry']])


--- Industry Distribution (Total 27) ---
predicted_industry
Real Estate    27
Name: count, dtype: int64

--- Detailed List ---
    business_id predicted_industry
0             2        Real Estate
1             3        Real Estate
2             4        Real Estate
3             5        Real Estate
4             6        Real Estate
5             7        Real Estate
6             8        Real Estate
7             9        Real Estate
8            10        Real Estate
9            11        Real Estate
10           12        Real Estate
11           13        Real Estate
12           14        Real Estate
13           15        Real Estate
14           16        Real Estate
15           17        Real Estate
16           18        Real Estate
17           19        Real Estate
18           20        Real Estate
19           21        Real Estate
20           22        Real Estate
21           23        Real Estate
22           24        Real Estate
23           25        Real Esta

In [ ]:
import pandas as pd

# 1. Load the Transactions Table (Where the actual activity is)
df_txn = dfs['Master_txn_table']

print("Analyzing Transaction Activity per Business...")

# 2. Define a function to score each business based on usage
def profile_business_activity(df, bid):
    # Filter for this business
    subset = df[df['business_id'] == bid]
    
    # Get the list of Accounts and Products actually used in transactions
    used_accounts = subset['account'].astype(str).str.lower().unique()
    used_products = subset['product_service'].astype(str).str.lower().unique()
    
    # Combine into one searchable text
    activity_text = " ".join(used_accounts) + " " + " ".join(used_products)
    
    # --- SCORING SYSTEM ---
    scores = {'Retail': 0, 'Service': 0, 'Real_Estate': 0}
    
    # Retail Signals (Inventory, Sales of Product)
    if any(x in activity_text for x in ['inventory', 'product', 'shipping', 'cogs', 'goods sold']):
        scores['Retail'] += 5
        
    # Service Signals (Consulting, Hours, Design)
    if any(x in activity_text for x in ['consulting', 'hours', 'service', 'design', 'legal']):
        scores['Service'] += 5
        
    # Real Estate Signals (Job Materials, Subcontractors, WIP)
    if any(x in activity_text for x in ['job materials', 'wip', 'subcontractor', 'concrete', 'roofing']):
        scores['Real_Estate'] += 5
        
    # Tie-Breaker Logic
    # If it uses "Job Materials", it's almost certainly Construction/Real Estate
    if 'job materials' in activity_text:
        return "Real_Estate"
    
    # Get the category with the highest score
    best_match = max(scores, key=scores.get)
    
    # If no score, return Undefined
    if scores[best_match] == 0:
        return "Undefined"
        
    return best_match

# 3. Analyze all 27 businesses
results = []
unique_bids = df_txn['business_id'].unique()

for bid in unique_bids:
    industry = profile_business_activity(df_txn, bid)
    results.append({'business_id': bid, 'industry': industry})

# 4. Convert to DataFrame and Show
df_results = pd.DataFrame(results)

print("\n--- TRUE Industry Distribution (Based on Transactions) ---")
print(df_results['industry'].value_counts()){
        "id": 3,
        "Query": "What was the first invoice for Matthew James?",
        "Level": "medium"
    }

print("\n--- Detailed Assignment ---")
print(df_results)

Analyzing Transaction Activity per Business...

--- TRUE Industry Distribution (Based on Transactions) ---
industry
Real_Estate    27
Name: count, dtype: int64

--- Detailed Assignment ---
    business_id     industry
0             2  Real_Estate
1             3  Real_Estate
2             4  Real_Estate
3             5  Real_Estate
4             6  Real_Estate
5             7  Real_Estate
6             8  Real_Estate
7             9  Real_Estate
8            10  Real_Estate
9            11  Real_Estate
10           12  Real_Estate
11           13  Real_Estate
12           14  Real_Estate
13           15  Real_Estate
14           16  Real_Estate
15           17  Real_Estate
16           18  Real_Estate
17           19  Real_Estate
18           20  Real_Estate
19           21  Real_Estate
20           22  Real_Estate
21           23  Real_Estate
22           24  Real_Estate
23           25  Real_Estate
24           26  Real_Estate
25           27  Real_Estate
26           28  Real_Estate

In [45]:
import pandas as pd

# 1. Load Transactions
df_txn = dfs['Master_txn_table']

print("Calculating Activity Ratios per Business...")

def get_activity_ratios(df, bid):
    subset = df[df['business_id'] == bid]
    total_rows = len(subset)
    
    if total_rows == 0:
        return None
    
    # Get all text columns to search
    text_data = subset['account'].astype(str).str.lower() + " " + \
                subset['product_service'].astype(str).str.lower()
    
    # 1. CONSTRUCTION SCORE (Heavy Materials)
    # Look for: job materials, concrete, roofing, wip
    construction_count = text_data.str.contains('job materials|concrete|roofing|wip|subcontractor').sum()
    
    # 2. SERVICE SCORE (Design, Consulting, Professional)
    # Look for: design, consulting, legal, professional, hours, labor
    service_count = text_data.str.contains('design|consulting|legal|professional|hours|labor').sum()
    
    # 3. HOSPITALITY/RETAIL SCORE (Hotels, Rent, Supplies)
    # Look for: hotel, casino, rent, supplies, inventory
    hospitality_count = text_data.str.contains('hotel|casino|rent|supplies|inventory').sum()
    
    return {
        'business_id': bid,
        'total_txns': total_rows,
        'Construction_Pct': round((construction_count / total_rows) * 100, 1),
        'Service_Pct': round((service_count / total_rows) * 100, 1),
        'Hospitality_Pct': round((hospitality_count / total_rows) * 100, 1)
    }

# Analyze all
ratios = []
for bid in df_txn['business_id'].unique():
    res = get_activity_ratios(df_txn, bid)
    if res:
        ratios.append(res)

df_ratios = pd.DataFrame(ratios)

# Sort by Service Pct to find the "Service-like" companies
print("\n--- Business Activity Mix (Sorted by Service %) ---")
print(df_ratios.sort_values(by='Service_Pct', ascending=False).head(10))

print("\n--- Business Activity Mix (Sorted by Hospitality %) ---")
print(df_ratios.sort_values(by='Hospitality_Pct', ascending=False).head(10))


Calculating Activity Ratios per Business...

--- Business Activity Mix (Sorted by Service %) ---
    business_id  total_txns  Construction_Pct  Service_Pct  Hospitality_Pct
12           14       30002               5.5          8.7              0.0
26           28       30003               5.4          4.4              0.0
0             2       30003               5.4          3.5              1.9
3             5       30003               5.4          3.5              1.9
4             6       30003               5.4          3.5              0.0
5             7       30003               5.4          3.5              0.0
6             8       30000               5.4          3.5              0.0
7             9       30000               5.4          3.5              0.0
8            10       30000               5.4          3.5              0.0
1             3       30000               5.4          3.5              3.7

--- Business Activity Mix (Sorted by Hospitality %) ---
    busine

In [ ]:
import pandas as pd
import os

# ================= CONFIGURATION =================
# The 3 IDs we identified from your analysis
TARGET_DOMAINS = {
    'Hospitality': 15,   # High Hospitality %
    'Service': 14,       # High Service %
    'Construction': 2    # Baseline Construction
}

OUTPUT_DIR = "./thesis_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================= EXTRACT AND SAVE =================
print(f"Extracting datasets for: {list(TARGET_DOMAINS.keys())}...\n")

# Load your cleaned dataframes (assuming 'dfs' dict exists from previous steps)
# If not re-load them:{
        "id": 3,
        "Query": "What was the first invoice for Matthew James?",
        "Level": "medium"
    }
# dfs['Master_txn_table'] = pd.read_csv('Master_txn_table.csv') ... etc

for domain, bid in TARGET_DOMAINS.items():
    print(f"Processing {domain} (Business ID: {bid})...")
    
    # Create folder
    domain_dir = os.path.join(OUTPUT_DIR, domain)
    os.makedirs(domain_dir, exist_ok=True)
    
    # Filter and Save ALL tables for this specific ID
    for table_name, df in dfs.items():
        # Check if table has business_id column
        if 'business_id' in df.columns:
            subset = df[df['business_id'] == bid]
            
            # Save to CSV
            # logic: ./thesis_datasets/Hospitality/Master_txn_table.csv
            save_path = os.path.join(domain_dir, f"{table_name}.csv")
            subset.to_csv(save_path, index=False)
            
    print(f"   -> Saved {len(dfs['Master_txn_table'][dfs['Master_txn_table']['business_id']==bid])} transactions.")

print("\n✅ DONE. You are ready to train/test your RAG system.")

Extracting datasets for: ['Hospitality', 'Service', 'Construction']...

Processing Hospitality (Business ID: 15)...
   -> Saved 30003 transactions.
Processing Service (Business ID: 14)...
   -> Saved 30002 transactions.
Processing Construction (Business ID: 2)...
   -> Saved 30003 transactions.

✅ DONE. You are ready to train/test your RAG system.


In [6]:
import pandas as pd
import os

# ================= CONFIGURATION =================
# Set path to your files
DATA_DIR = "./Tables" 

# ================= 1. LOAD DATA =================
print("Loading files...")

# Load the NEW Industry Details file
df_details = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry_Details.csv")
# Clean column names
df_details.columns = df_details.columns.str.strip().str.lower().str.replace(' ', '_')

# Load the Chart of Accounts (which has Business IDs)
df_coa = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/chart_of_account_OB.csv")
# Clean column names
df_coa = df_coa.loc[:, ~df_coa.columns.str.contains('^Unnamed')]
df_coa.columns = df_coa.columns.str.strip().str.lower().str.replace(' ', '_')

# ================= 2. MAP BUSINESS ID TO INDUSTRY =================
print("Mapping Business IDs to Industries...")

# We merge on 'account_name'. 
# This finds the Industry for every account in every business.
merged = pd.merge(
    df_coa, 
    df_details[['industry', 'account_name']], 
    on='account_name', 
    how='inner'
)

# Now we count which Industry appears most often for each Business ID
# (This handles generic accounts like "Bookkeeper" that might appear in multiple)
industry_counts = merged.groupby(['business_id', 'industry']).size().reset_index(name='count')

# Get the Top Industry for each Business ID
final_mapping = industry_counts.sort_values(['business_id', 'count'], ascending=[True, False]) \
    .drop_duplicates(['business_id'])

print("\n--- GROUND TRUTH INDUSTRY MAPPING ---")
print(final_mapping[['business_id', 'industry']].to_string(index=False))

# ================= 3. SELECT THESIS CANDIDATES =================
# We need to map these detailed industries to your 3 Thesis Categories
# Retail, Construction, Service

print("\nSelecting Thesis Candidates...")

candidates = {}

for idx, row in final_mapping.iterrows():
    bid = row['business_id']
    ind = row['industry']
    
    # MAPPING LOGIC
    
    # 1. Retail (Online Retail, Retail Trade)
    if 'Retail' in ind and 'Retail' not in candidates:
        candidates['Retail'] = bid
        print(f"✅ Found Retail: ID {bid} ({ind})")

    # 2. Construction (Construction, Specialist Engineering)
    elif ('Construction' in ind or 'Contractors' in ind) and 'Construction' not in candidates:
        candidates['Construction'] = bid
        print(f"✅ Found Construction: ID {bid} ({ind})")
        
    # 3. Service (Professional Services, Admin, Info)
    # We prefer 'Professional, Scientific' or 'Information'
    elif ('Professional' in ind or 'Information' in ind or 'Advisory' in ind) and 'Service' not in candidates:
        candidates['Service'] = bid
        print(f"✅ Found Service: ID {bid} ({ind})")

# Avoid calling .head() on the return value of print() which is None
if candidates:
    print("\nFinal Candidates for Thesis:", candidates)
else:
    print("\nNo final candidates for thesis were found.")

candidates_df = pd.DataFrame(list(candidates.items()), columns=['Thesis Category', 'Business ID'])
print(candidates_df.head(30))

Loading files...
Mapping Business IDs to Industries...

--- GROUND TRUTH INDUSTRY MAPPING ---
 business_id                                                       industry
           2                                Accommodation and Food Services
           3 Administration, Business Support and Waste Management Services
           4                     Agriculture, Forestry, Fishing and Hunting
           5                             Arts, Entertainment and Recreation
           6                                                   Construction
           7                                           Educational Services
           8                                          Finance and Insurance
           9                               Healthcare and Social Assistance
          10                                                    Information
          11                                                  Manufacturing
          12                                                         M

In [47]:
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

INDUSTRY_DETAILS_PATH = (
    "/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry_Details.csv"
)

COA_PATH = (
    "/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/chart_of_account_OB.csv"
)

PRODUCT_SERVICE_PATH = (
    "/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/product_service_table.csv"
)

OUTPUT_MAPPING_PATH = "business_id_to_industry_context.csv"
OUTPUT_CANDIDATES_PATH = "thesis_business_candidates.csv"

# ============================================================
# UTILITIES
# ============================================================

def clean_columns(df):
    df = df.loc[:, ~df.columns.str.contains("^unnamed", case=False)]
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

def normalize_text(series):
    return series.astype(str).str.strip().str.lower()

# ============================================================
# 1. LOAD DATA
# ============================================================

print("Loading datasets...")

df_details = pd.read_csv(INDUSTRY_DETAILS_PATH)
df_details = clean_columns(df_details)

df_coa = pd.read_csv(COA_PATH)
df_coa = clean_columns(df_coa)

df_ps = pd.read_csv(PRODUCT_SERVICE_PATH)
df_ps = clean_columns(df_ps)

# Normalize relevant text columns
df_details["account_name"] = normalize_text(df_details["account_name"])
df_details["product_service"] = normalize_text(df_details["product_service"])

df_coa["account_name"] = normalize_text(df_coa["account_name"])
df_ps["product_service"] = normalize_text(df_ps["product_service"])

# ============================================================
# 2. BUILD EVIDENCE
# ============================================================

print("Building industry evidence...")

# ---------- Account-based evidence ----------
account_evidence = pd.merge(
    df_coa,
    df_details[["industry", "account_name"]],
    on="account_name",
    how="inner"
)[["business_id", "industry"]]

account_evidence["source"] = "account"

# ---------- Service-based evidence ----------
service_evidence = pd.merge(
    df_ps,
    df_details[["industry", "product_service"]],
    on="product_service",
    how="inner"
)[["business_id", "industry"]]

service_evidence["source"] = "service"

# Combine evidence
combined_evidence = pd.concat(
    [account_evidence, service_evidence],
    ignore_index=True
)

# ============================================================
# 3. AGGREGATE → WEAKLY INFER INDUSTRY
# ============================================================

industry_counts = (
    combined_evidence
    .groupby(["business_id", "industry"])
    .size()
    .reset_index(name="evidence_count")
)

dominant_industry = (
    industry_counts
    .sort_values(
        ["business_id", "evidence_count"],
        ascending=[True, False]
    )
    .groupby("business_id")
    .first()
    .reset_index()
)

total_evidence = (
    industry_counts
    .groupby("business_id")["evidence_count"]
    .sum()
    .reset_index(name="total_evidence")
)

dominant_industry = dominant_industry.merge(
    total_evidence,
    on="business_id"
)

dominant_industry["confidence"] = (
    dominant_industry["evidence_count"]
    / dominant_industry["total_evidence"]
).round(2)

dominant_industry = dominant_industry[
    ["business_id", "industry", "confidence"]
]

dominant_industry.to_csv(OUTPUT_MAPPING_PATH, index=False)

print("\n--- WEAKLY INFERRED INDUSTRY CONTEXT ---")
print(dominant_industry.sort_values("business_id").to_string(index=False))

# ============================================================
# 4. SELECT THESIS CANDIDATES (CONTROLLED, MANUAL LOGIC)
# ============================================================

print("\nSelecting representative thesis businesses...")

candidates = []

for _, row in dominant_industry.iterrows():
    bid = row["business_id"]
    ind = row["industry"]

    if "Retail" in ind and not any(c[0] == "Retail" for c in candidates):
        candidates.append(("Retail", bid, ind))
        print(f"✅ Retail → Business {bid} ({ind})")

    elif (
        ("Construction" in ind or "Contractors" in ind)
        and not any(c[0] == "Construction" for c in candidates)
    ):
        candidates.append(("Construction", bid, ind))
        print(f"✅ Construction → Business {bid} ({ind})")

    elif (
        ("Professional" in ind or "Information" in ind or "Advisory" in ind)
        and not any(c[0] == "Service" for c in candidates)
    ):
        candidates.append(("Service", bid, ind))
        print(f"✅ Service → Business {bid} ({ind})")

df_candidates = pd.DataFrame(
    candidates,
    columns=["thesis_category", "business_id", "industry"]
)

df_candidates.to_csv(OUTPUT_CANDIDATES_PATH, index=False)

print("\n--- FINAL THESIS BUSINESS SELECTION ---")
print(df_candidates.to_string(index=False))

print("\n✔ Industry context mapping completed.")


Loading datasets...
Building industry evidence...

--- WEAKLY INFERRED INDUSTRY CONTEXT ---
 business_id                                                       industry  confidence
           2                                Accommodation and Food Services        1.00
           3 Administration, Business Support and Waste Management Services        0.96
           4                     Agriculture, Forestry, Fishing and Hunting        0.92
           5                             Arts, Entertainment and Recreation        0.88
           6                                                   Construction        1.00
           7                                           Educational Services        0.96
           8                                          Finance and Insurance        1.00
           9                               Healthcare and Social Assistance        1.00
          10                                                    Information        1.00
          11                

In [ ]:
# convert xls to csv
import pandas as pd/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/product_service_table
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
df_details= pd.read_excel('/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry Details.xlsx')
df_details.to_csv('/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry_Details.csv', index=False)


In [3]:
df_details.head(30)

,Industry,Business,Account Name,Account Type,Product_Service,Product_Service_Type
0,Accommodation and Food Services,Accommodation and Food Services Industry,Hotels and motel services,Income,Casino Hotels,Service
1,Accommodation and Food Services,Accommodation and Food Services Industry,Campground and RV park services,Income,Camping and Parking,Service
2,Accommodation and Food Services,Accommodation and Food Services Industry,Other accommodation services,Income,Traveller accomodation,Service
3,Accommodation and Food Services,Accommodation and Food Services Industry,Full-service restaurants,Income,Drinking and Special Food,Service
4,Accommodation and Food Services,Accommodation and Food Services Industry,Fast food restaurants,Income,Fast food,Service
5,Accommodation and Food Services,Accommodation and Food Services Industry,Coffee and snack shops,Income,Coffee and Snack,Service
6,Accommodation and Food Services,Accommodation and Food Services Industry,Catering services,Income,Catering,Service
7,Accommodation and Food Services,Accommodation and Food Services Industry,Street vending locations,Income,Street vendoring,Service
8,Accommodation and Food Services,Accommodation and Food Services Industry,Bars and nightclubs,Income,Bars and Nightclubs,Service
9,Accommodation and Food Services,Accommodation and Food Services Industry,Other food services,Income,Foods on demand,Service


In [ ]:
import pandas as pd

# -----------------------------
# 1. Load datasets
# -----------------------------
industry_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/BookSQL_Generation/data/Industry_Details.csv")
coa_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/chart_of_account_OB.csv")
ps_df = pd.read_csv("/home/bisag/Documents/Devanshi_Intern/BookSQL/DATA/BookSQL/Tables/product_service_table.csv")

# Normalize text
def normalize(text):
    return str(text).strip().lower()

industry_df["Account Name"] = industry_df["Account Name"].apply(normalize)
industry_df["Product_Service"] = industry_df["Product_Service"].apply(normalize)

coa_df["Account name"] = coa_df["Account name"].apply(normalize)
ps_df["Product_Service"] = ps_df["Product_Service"].apply(normalize)

# -----------------------------
# 2. Match via Accounts
# -----------------------------
account_matches = coa_df.merge(
    industry_df[["Industry", "Account Name"]],
    left_on="Account name",
    right_on="Account Name",
    how="inner"
)[["Business Id", "Industry"]]

account_matches["source"] = "account"

# -----------------------------
# 3. Match via Product / Service
# -----------------------------
ps_matches = ps_df.merge(
    industry_df[["Industry", "Product_Service"]],
    on="Product_Service",
    how="inner"
)[["Business Id", "Industry"]]

ps_matches["source"] = "product_service"

# -----------------------------
# 4. Combine Evidence
# -----------------------------
combined = pd.concat([account_matches, ps_matches], ignore_index=True)

# -----------------------------
# 5. Aggregate → Find dominant industry per Business Id
# -----------------------------
industry_scores = (
    combined
    .groupby(["Business Id", "Industry"])
    .size()
    .reset_index(name="evidence_count")
)

dominant_industry = (
    industry_scores
    .sort_values(["Business Id", "evidence_count"], ascending=[True, False])
    .groupby("Business Id")
    .first()
    .reset_index()
)

# -----------------------------
# 6. Optional: Confidence score
# -----------------------------
total_counts = (
    industry_scores
    .groupby("Business Id")["evidence_count"]
    .sum()
    .reset_index(name="total")
)

dominant_industry = dominant_industry.merge(total_counts, on="Business Id")
dominant_industry["confidence"] = (
    dominant_industry["evidence_count"] / dominant_industry["total"]
).round(2)

# -----------------------------
# 7. Final Mapping
# -----------------------------
business_industry_map = dominant_industry[
    ["Business Id", "Industry", "confidence"]
]

business_industry_map.to_csv("business_id_to_industry.csv", index=False)

print("✅ Business → Industry mapping generated")
